<a href="https://colab.research.google.com/github/AhmedAlwaqidi/meeting_summariser_model/blob/main/meeting_summariser_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Set-up working packages

In [7]:
!pip install -q faster-whisper transformers sentencepiece accelerate

Upload the meeting audio

In [2]:
from google.colab import files

uploaded = files.upload()

audio_file = list(uploaded.keys())[0]

print("Uploaded:", audio_file)

Saving m_test.m4a to m_test (1).m4a
Uploaded: m_test (1).m4a


Transcribe using whisper model

In [3]:
from faster_whisper import WhisperModel

model = WhisperModel(
    "small",
    device="cuda",
    compute_type="float16"
)

segments, info = model.transcribe(
    audio_file,
    beam_size=5
)

transcript_segments = []

for segment in segments:
    transcript_segments.append({
        "start": segment.start,
        "end": segment.end,
        "text": segment.text.strip()
    })

print("Detected language:", info.language)
print("Language probability:", info.language_probability)

for segment in transcript_segments:
    print(

        f"[{segment['start']:.2f} - {segment['end']:.2f}] "
        f"{segment['text']}"
    )

Detected language: ar
Language probability: 0.43994140625
[0.00 - 12.00] مرحباً أحمد . أنا سأتحدث اليوم . أنا سأتحدث اليوم about the project that we have to do
[12.00 - 26.00] اليوم سوف نتحدث عن الأشياء التي ستكون أسائلت لكم . مصر أمير ومصر أيمان
[27.00 - 39.00] يمكنك أن تتحدث عن الأشياء ولكن يمكنك أن تتحدث عن الأشياء . هل هناك سؤال عن الأشياء ؟
[41.00 - 49.00] مرحباً أنا أيمان . لا أستطيع أن أتحدث عن الأشياء الآن لأن أنا مصر أمير
[49.00 - 67.00] لذا يمكنك أن تتحدث عن الأشياء . أمير أنا . حسناً لذا لا تستطيع أن تفعل ما يريدون . لذا أتحدث عن الأشياء ونفعل ذلك مع الأشياء . بالتأكيد
[67.00 - 78.00] هناك سؤال عن الأشياء . أنا أم زيامد . ويجب أن أتحدث عن الأشياء . ويجب أن أتحدث عن الأشياء


Save the transcript

In [4]:
transcript = "\n".join(
    segment["text"]
    for segment in transcript_segments
)


#merge the transcripts with timestamps
for segment in transcript_segments:
    print(
        f"[{segment['start']:.2f}s - {segment['end']:.2f}s] "
        f"{segment['text']}"
    )

#same the transcript to a file
import json

with open("meeting_transcript.json", "w", encoding="utf-8") as f:
    json.dump(
        transcript_segments,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Transcript saved successfully.")

[0.00s - 12.00s] مرحباً أحمد . أنا سأتحدث اليوم . أنا سأتحدث اليوم about the project that we have to do
[12.00s - 26.00s] اليوم سوف نتحدث عن الأشياء التي ستكون أسائلت لكم . مصر أمير ومصر أيمان
[27.00s - 39.00s] يمكنك أن تتحدث عن الأشياء ولكن يمكنك أن تتحدث عن الأشياء . هل هناك سؤال عن الأشياء ؟
[41.00s - 49.00s] مرحباً أنا أيمان . لا أستطيع أن أتحدث عن الأشياء الآن لأن أنا مصر أمير
[49.00s - 67.00s] لذا يمكنك أن تتحدث عن الأشياء . أمير أنا . حسناً لذا لا تستطيع أن تفعل ما يريدون . لذا أتحدث عن الأشياء ونفعل ذلك مع الأشياء . بالتأكيد
[67.00s - 78.00s] هناك سؤال عن الأشياء . أنا أم زيامد . ويجب أن أتحدث عن الأشياء . ويجب أن أتحدث عن الأشياء
Transcript saved successfully.


Load model well do the summarization

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM loaded successfully.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully.


Ask the LLM model to analyze the meeting

In [11]:
prompt = f"""
You are a professional meeting analysis assistant.

Analyze the following meeting transcript.

Provide the result in English using this structure:

1. Meeting Summary
2. Key Topics
3. Important Points
4. Decisions Made
5. Action Items

For action items, include:
- Task
- Person responsible, if mentioned
- Deadline, if mentioned

Do not invent information.
If something was not mentioned, say "Not specified".

Meeting transcript:

{transcript}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(llm.device)

outputs = llm.generate(
    **inputs,
    max_new_tokens=1000,
    temperature=0.2,
    do_sample=True
)

summary = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(summary)

### Meeting Summary
The meeting is discussing a project and involves three participants: Ahmed, Amir, and Ameen. The conversation revolves around what topics will be discussed during the meeting.

### Key Topics
- Project discussion
- Topics to be covered in the meeting

### Important Points
- Amir is unable to speak at the moment due to being busy with another task (presumably Egypt Amir).
- Amir has been asked to discuss the topics.
- Ameen needs to speak about the topics as well.

### Decisions Made
- No decisions were made during the meeting.

### Action Items
- **Task:** Discuss the project topics.
  - **Person Responsible:** Amir and Ameen
  - **Deadline:** Not specified

- **Task:** Ensure all topics are covered.
  - **Person Responsible:** Amir and Ameen
  - **Deadline:** Not specified

- **Task:** Amir should speak about the topics.
  - **Person Responsible:** Amir
  - **Deadline:** Not specified

- **Task:** Ameen should speak about the topics.
  - **Person Responsible:** Ame